# Compression results

**Purpose:** Present token-count compression results from committed evaluation tables and bootstrap summaries.

**Inputs (committed result/provenance files read):**
- [`results/tables/eval_aggregate.csv`](../results/tables/eval_aggregate.csv)
- [`results/processed/bootstrap.json`](../results/processed/bootstrap.json)
- [`results/tables/fig01_primary_reduction.csv`](../results/tables/fig01_primary_reduction.csv)
- [`results/tables/fig02_tradeoff.csv`](../results/tables/fig02_tradeoff.csv)
- [`results/tables/fig03_transfer.csv`](../results/tables/fig03_transfer.csv)

**Outputs:** Rendered compression tables in this notebook only; no files are written.

**Expected runtime:** ~4 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Compression tables

Authoritative files include [`eval_aggregate.csv`](../results/tables/eval_aggregate.csv) and [`bootstrap.json`](../results/processed/bootstrap.json). Claims trace to the claim ledger.

In [2]:
eval_agg = load_csv('results/tables/eval_aggregate.csv')
bootstrap = load_json('results/processed/bootstrap.json')
primary = load_csv('results/tables/fig01_primary_reduction.csv')
tradeoff = load_csv('results/tables/fig02_tradeoff.csv')
transfer = load_csv('results/tables/fig03_transfer.csv')

In [3]:
if not eval_agg.empty:
    principal = eval_agg[eval_agg['eval_set'].eq('azure_cross_service_holdout')].copy()
    cols = ['tokenizer_id', 'n_documents', 'tokens_per_1000_utf8_bytes_micro', 'chars_per_token', 'roundtrip_ok_rate', 'unk_total']
    display(principal[cols].sort_values(['tokenizer_id']))
else:
    note_missing('results/tables/eval_aggregate.csv')

,tokenizer_id,n_documents,tokens_per_1000_utf8_bytes_micro,chars_per_token,roundtrip_ok_rate,unk_total
0,azure-bytebpe-16384,964,240.180213,4.161165,1.0,0
1,azure-bytebpe-32768,964,232.776060,4.293524,1.0,0
2,azure-bytebpe-8192,964,256.210308,3.900817,1.0,0
3,general-bytebpe-16384,964,379.295492,2.634963,1.0,0
4,general-bytebpe-32768,964,360.377429,2.773286,1.0,0
5,general-bytebpe-8192,964,403.225451,2.478588,1.0,0
6,mixed-bytebpe-16384,964,249.523101,4.005359,1.0,0
7,mixed-bytebpe-32768,964,238.815804,4.184939,1.0,0
8,mixed-bytebpe-8192,964,270.921452,3.689001,1.0,0
9,tiktoken-cl100k_base,964,217.391601,4.597370,1.0,0


In [4]:
if not primary.empty:
    display(primary)
else:
    note_missing('results/tables/fig01_primary_reduction.csv')

,vocab,reduction_pct,ci_low,ci_high
0,8192,36.459788,35.591666,37.332441
1,16384,36.677282,35.767853,37.618229
2,32768,35.407703,34.451369,36.408188


In [5]:
for label, df in [('tradeoff', tradeoff), ('transfer', transfer)]:
    print(label)
    if not df.empty:
        display(df.head(20))
    else:
        note_missing(f'results/tables/fig0*_{{label}}.csv')

tradeoff


,tokenizer,azure_holdout_pct,general_english_pct
0,General,0.000000,0.000000
1,Azure,36.677282,-34.747447
2,Mixed,34.214061,-4.394185


transfer


,eval_set,label,reduction_pct,ci_low,ci_high
0,azure_cross_service_holdout,Azure cross-service,36.677282,35.767853,37.618229
1,azure_same_service_holdout,Azure same-service,35.025652,34.062315,36.032467
2,ms_data,MS data,32.936548,31.331402,34.545389
3,ms_devops,MS DevOps,31.758530,30.841347,32.668159
4,external_technical,Kubernetes (external),31.177245,29.909112,32.132089
5,ms_architecture,MS architecture,26.029448,25.202611,26.829502
6,ms_broad_enterprise,MS broad enterprise,25.885464,24.467217,27.324929
7,general_english,General English,-34.747447,-35.102541,-34.398337
